In [2]:
from db_cloud_func import *
import sqlite3
import pandas as pd
# OXY

========| ('irfan_admin', 'trade_app') |========
*** ✅ SUCCESSFUL CLOUD CONNECTION ⛓️ ***


In [3]:
local_db_path = 'trade_ai.db'

In [4]:
def read_distinct_symbols(db_path):
    query = "SELECT DISTINCT SYMBOL FROM usa_1min_polygon_past"
    
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(query, conn)
    
    print(f"Distinct symbol count: {len(df)}")
    return df


# kullanım
df_symbols = read_distinct_symbols(local_db_path)

Distinct symbol count: 425


In [7]:
def read_table_as_df(db_path, table_name, SYMBOL):
    conn = sqlite3.connect(db_path)
    
    try:
        query = f"SELECT * FROM {table_name} WHERE SYMBOL = ?"
        df = pd.read_sql_query(query, conn, params=(SYMBOL,))
        
    finally:
        conn.close()
    
    # print(f'Table count {len(df)}')
    return df

In [10]:
def filter_from_high(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """
    SYMBOL'a göre filtreler.
    HIGH maksimum satırı bulur.
    
    Eğer max HIGH tarihi son 365 gün içindeyse:
        -> filtre başlangıcı = bugünden 365 gün önce
    Eğer max HIGH tarihi 365 günden eskiyse:
        -> filtre başlangıcı = max HIGH gününün başı (00:00:00)
    
    Sonuç: belirlenen başlangıçtan son tarihe kadar filtrelenmiş df.
    """
    
    temp = df.copy()
    temp = temp[temp["SYMBOL"] == symbol]

    if temp.empty:
        print(f"SYMBOL: {symbol} | Veri bulunamadı.")
        return temp

    temp["TIMESTAMP"] = pd.to_datetime(temp["TIMESTAMP"])
    temp = temp.sort_values("TIMESTAMP", ascending=True)

    # Max HIGH satırı
    idx_max = temp["HIGH"].idxmax()
    max_high = temp.loc[idx_max, "HIGH"]
    max_timestamp = temp.loc[idx_max, "TIMESTAMP"]

    # Gün başlangıcı
    max_day_start = max_timestamp.normalize()

    # Son 365 gün hesabı
    now = pd.Timestamp.now()
    one_year_ago = now - pd.Timedelta(days=365)

    # Koşul
    if max_timestamp >= one_year_ago:
        filter_start = one_year_ago.normalize()
        rule_used = "Son 365 gün kuralı kullanıldı"
    else:
        filter_start = max_day_start
        rule_used = "Max HIGH tarihi baz alındı"

    # Filtreleme
    result = temp[temp["TIMESTAMP"] >= filter_start].reset_index(drop=True)

    # Tek print
    print(
        f"SYMBOL: {symbol} | "
        f"En Yüksek HIGH: {max_high} | "
        f"High Zamanı: {max_timestamp.strftime('%Y-%m-%d %H:%M:%S')} | "
        f"Filtre Başlangıç: {filter_start.strftime('%Y-%m-%d %H:%M:%S')} | "
        f"Kural: {rule_used} | "
        f"Kalan Satır: {len(result)}")

    return result

In [12]:
for SYMBOL in df_symbols['SYMBOL'].to_list():

    print(f'➡️ {SYMBOL}')
    
    df = read_table_as_df(
        db_path=local_db_path,
        table_name='usa_1min_polygon_past',
        SYMBOL=SYMBOL)

    filtered_df = filter_from_high(df, SYMBOL)
    fn_write_cloud(df, schema='source', name='usa_1min_high_filtered', if_exists='append',is_only_distinct = False,dist_col=None)

➡️ MMM
SYMBOL: MMM | En Yüksek HIGH: 177.41 | High Zamanı: 2026-02-12 17:36:00 | Filtre Başlangıç: 2025-02-23 00:00:00 | Kural: Son 365 gün kuralı kullanıldı | Kalan Satır: 97718
➡️ AOS
SYMBOL: AOS | En Yüksek HIGH: 92.445 | High Zamanı: 2024-07-18 14:05:00 | Filtre Başlangıç: 2024-07-18 00:00:00 | Kural: Max HIGH tarihi baz alındı | Kalan Satır: 139070
➡️ ABT
SYMBOL: ABT | En Yüksek HIGH: 141.23 | High Zamanı: 2025-03-04 14:30:00 | Filtre Başlangıç: 2025-02-23 00:00:00 | Kural: Son 365 gün kuralı kullanıldı | Kalan Satır: 98998
➡️ ABBV
SYMBOL: ABBV | En Yüksek HIGH: 244.81 | High Zamanı: 2025-10-01 19:59:00 | Filtre Başlangıç: 2025-02-23 00:00:00 | Kural: Son 365 gün kuralı kullanıldı | Kalan Satır: 100238
➡️ ACN
SYMBOL: ACN | En Yüksek HIGH: 398.35 | High Zamanı: 2025-02-05 20:59:00 | Filtre Başlangıç: 2025-02-05 00:00:00 | Kural: Max HIGH tarihi baz alındı | Kalan Satır: 102954
➡️ AYI
SYMBOL: AYI | En Yüksek HIGH: 380.1699 | High Zamanı: 2026-01-05 20:20:00 | Filtre Başlangıç: 2025-